In [1]:
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import time, random
from requests.exceptions import RequestException, Timeout

ques_url_list = [f'http://club.xywy.com/list_all_{i}.htm' for i in range(1,1001)]
output_file = 'output.json'


prefix_url = 'http://club.xywy.com'

def get_soup(url, timeout=10):
    # 创建一个UserAgent对象，用于随机生成User-Agent头
    ua = UserAgent()
    # 定义爬取目标URL和请求头
    headers = {
        'User-Agent': ua.random,
        'Connection': 'keep-alive'
    }

    time.sleep(random.uniform(0.5, 1.5))

    # 发送请求，获取响应
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
    except (RequestException, Timeout):
        print(url, 'status code is: ', response.status_code)
        return None
    # 解析HTML文本
    try:
        soup = BeautifulSoup(response.content, 'html.parser', from_encoding='utf-8')
    except AttributeError:
        print("Response content attribute error!!!")
        return None
    return soup


def get_ans_info_good(ques_ans_url):
    soup = get_soup(ques_ans_url)
    ans_text = '\n'.join([a.text.strip() for a in soup.select('.replay-content-box')])
    info_text = '  '.join([a.text.strip() for a in soup.select('.doc-txt span')])
    good_text = soup.select_one('.doc-goodat').text.strip()
    return ans_text, info_text, good_text

def get_head_url(ques_url):
    
    soup = get_soup(ques_url)
    ret_list = []
    for th in soup.select('.th'):
        ques_ans_url = prefix_url + th['href']
        ques_text = th.text.strip()
        ans_text, info_text, good_text = get_ans_info_good(ques_ans_url)
        ret_list.append([ques_text, ques_ans_url, ans_text, info_text, good_text])
    return ret_list


final_list = []

for ques_url in ques_url_list:
    final_list.extend(get_head_url(ques_url))

KeyboardInterrupt: 